# Normalized PSAMA Statistical Rigor and Error Analysis

This demo notebook evaluates normalized predictive state adaptive moving average (PSAMA) against static moving averages and naive persistence across Ornstein-Uhlenbeck stochastic process trials. We compute error metrics (MSE, RMSE, MAE) and Wilcoxon signed-rank paired statistical significance tests.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'loguru==0.7.3')

In [ ]:
import json
import os
import urllib.request
from pathlib import Path
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-4b74fb-self-normalized-phase-space-adaptive-mov/main/round-2/evaluation-1/demo/mini_demo_data.json"

def load_data():
    try:
        logger.info(f"Trying to load data from GitHub URL: {GITHUB_DATA_URL}")
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        logger.warning(f"Failed to load from GitHub: {e}. Falling back to local file.")
    
    local_path = "mini_demo_data.json"
    if os.path.exists(local_path):
        logger.info(f"Loading data from local file: {local_path}")
        with open(local_path) as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local disk.")

data = load_data()

## Configuration and Parameter Setup

We define configuration parameters for our evaluation.

In [ ]:
# Configuration parameters
MAX_EXAMPLES = 50  # Number of examples to process from dataset
logger.info(f"Configuration set: MAX_EXAMPLES={MAX_EXAMPLES}")

## Evaluation Processing and Metrics Computation

Extract actuals and predictions, compute MSE, RMSE, and MAE for adaptive moving average, static moving average, and naive persistence baselines.

In [ ]:
logger.info("Starting evaluation of adaptive moving average forecasting vs baselines.")

dep_examples = data["datasets"][0]["examples"][:MAX_EXAMPLES]
logger.info(f"Loaded {len(dep_examples)} examples for evaluation.")

actuals = []
pred_adap = []
pred_stat = []
pred_naiv = []

for ex in dep_examples:
    actuals.append(float(ex["output"]))
    pred_adap.append(float(ex["predict_adaptive_ma"]))
    pred_stat.append(float(ex["predict_static_ma"]))
    pred_naiv.append(float(ex["predict_naive"]))
    
actuals = np.array(actuals)
pred_adap = np.array(pred_adap)
pred_stat = np.array(pred_stat)
pred_naiv = np.array(pred_naiv)

def compute_metrics(y_true, y_pred):
    mse = float(np.mean((y_true - y_pred) ** 2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(y_true - y_pred)))
    return mse, rmse, mae
    
mse_adap, rmse_adap, mae_adap = compute_metrics(actuals, pred_adap)
mse_stat, rmse_stat, mae_stat = compute_metrics(actuals, pred_stat)
mse_naiv, rmse_naiv, mae_naiv = compute_metrics(actuals, pred_naiv)

# Statistical tests (Wilcoxon signed-rank test on squared errors)
se_adap = (actuals - pred_adap) ** 2
se_stat = (actuals - pred_stat) ** 2
se_naiv = (actuals - pred_naiv) ** 2

wilcoxon_stat_vs_adap = stats.wilcoxon(se_stat, se_adap)
wilcoxon_naiv_vs_adap = stats.wilcoxon(se_naiv, se_adap)

logger.info(f"MSE Adaptive: {mse_adap:.4f}, Static: {mse_stat:.4f}, Naive: {mse_naiv:.4f}")

## Results Summary and Visualization

Display aggregate metrics and plot the forecast trajectories against actual series values.

In [ ]:
print("="*60)
print("EVALUATION RESULTS SUMMARY")
print("="*60)
print(f"Adaptive MA  - MSE: {mse_adap:.4f} | RMSE: {rmse_adap:.4f} | MAE: {mae_adap:.4f}")
print(f"Static MA    - MSE: {mse_stat:.4f} | RMSE: {rmse_stat:.4f} | MAE: {mae_stat:.4f}")
print(f"Naive Pers.  - MSE: {mse_naiv:.4f} | RMSE: {rmse_naiv:.4f} | MAE: {mae_naiv:.4f}")
print("-"*60)
print(f"Wilcoxon Static vs Adaptive p-value: {wilcoxon_stat_vs_adap.pvalue:.4e}")
print(f"Wilcoxon Naive vs Adaptive p-value: {wilcoxon_naiv_vs_adap.pvalue:.4e}")
print("="*60)

plt.figure(figsize=(10, 5))
plt.plot(actuals, label="Actual", color="black", linewidth=2)
plt.plot(pred_adap, label="Adaptive MA", color="blue", linestyle="--")
plt.plot(pred_stat, label="Static MA", color="green", linestyle="-.")
plt.plot(pred_naiv, label="Naive Persistence", color="orange", alpha=0.7)
plt.title("Ornstein-Uhlenbeck Forecasting Comparison")
plt.xlabel("Step Index")
plt.ylabel("Value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
